# Chapter 14: Release Automation: Tags, Changelogs, Packages (notebook edition)

## Learning Objectives

- Parse and order SemVer
- Verify release assets
- Explain notes, tag triggers and immutability

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. SemVer rules

Strict parsing, numeric ordering, pre-releases below releases.

In [ ]:
from intro_gha.semver import parse, latest, tag_to_version
for bad in ["1.0", "v1.0.0", "01.0.0"]:
    try: parse(bad); print("accepted?!", bad)
    except ValueError as e: print("rejected:", bad)
print(sorted(["2.0.0", "2.0.0-beta.1", "1.9.10", "1.9.9"], key=lambda v: parse(v).sort_key()))
assert latest(["0.1.9", "0.1.10"]) == "0.1.10" and latest(["1.0.0-rc.1", "1.0.0"]) == "1.0.0"

## 2. The v0.1.0 release assets

GitHub's digests, the local hashes and `SHA256SUMS` agreed. We also check the size of the checksum file by hand.

In [ ]:
import json
s = json.loads((ROOT/"fixtures"/"release_ch14_summary.json").read_text())
assets = {a["name"]: a for a in s["tag_push"]["release"]["assets"]}
for n, a in assets.items(): print(f"{n:38} {a['size']:>5} {a['digest'][:19]}")
line = lambda f: 64 + 2 + len("./" + f) + 1
total = line("tally_demo-0.1.0-py3-none-any.whl") + line("tally_demo-0.1.0.tar.gz")
print("SHA256SUMS bytes:", total)
assert total == assets["SHA256SUMS"]["size"] == 194

## 3. What started (and what did not)

A user-pushed tag started the release plus two `paths`-filtered workflows; the `GITHUB_TOKEN`-created tag started nothing.

In [ ]:
print(s["tag_push"]["other_workflows_fired_on_the_tag"])
print("token-created tag push runs:", s["dispatch"]["push_runs_started_for_tag"])
assert s["dispatch"]["push_runs_started_for_tag"] == 0 and len(s["tag_push"]["other_workflows_fired_on_the_tag"]) == 2

## 4. Generated notes

Only merged PRs are listed, grouped by label; the base tag matters.

In [ ]:
n = s["notes"]["preview_for_v0.1.0"]
print(n["categories"], n["lists_prs"], "direct commits listed:", n["direct_commits_listed"])
print("auto notes repeated PR #2:", s["dispatch"]["auto_notes_repeated_pr_2"], "| fixed by notes_start_tag:", s["notes_start_tag"]["notes_start_tag"])
assert not n["direct_commits_listed"] and s["dispatch"]["auto_notes_repeated_pr_2"]

## 5. Immutable releases

Baseline edits worked; after publishing under immutability they were refused.

In [ ]:
im = s["immutable"]
print(im["baseline"]); print(im["upload_extra_asset"]); print(im["delete_asset"]); print(im["delete_tag"])
print(im["after_disabling_setting"])
assert im["after_disabling_setting"] == {"v0.1.3": True, "v0.1.2": False}

## Takeaways & Next Steps

- Validate versions in code; build once; checksum.
- Pass `--notes-start-tag`; notes list merged PRs only.
- Tokens create tags that trigger nothing.
- Next: Phase 4, Chapter 15 on `GITHUB_TOKEN` and permissions.

## Chapter Link

Read: `learning_modules/chapter_14_release_automation.md`